In [ ]:
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  !pip install tf-nightly
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
# Load the tab-separated data. Each row contains the label followed by the SMS text.
column_names = ["label", "message"]
train_df = pd.read_csv(
    train_file_path,
    sep="\t",
    header=None,
    names=column_names,
    quoting=3,
    keep_default_na=False
)
valid_df = pd.read_csv(
    test_file_path,
    sep="\t",
    header=None,
    names=column_names,
    quoting=3,
    keep_default_na=False
)

# Remove empty rows and map ham -> 0, spam -> 1.
label_to_number = {"ham": 0, "spam": 1}
train_df["label"] = train_df["label"].str.strip().str.lower()
valid_df["label"] = valid_df["label"].str.strip().str.lower()
train_df = train_df[train_df["label"].isin(label_to_number)].copy()
valid_df = valid_df[valid_df["label"].isin(label_to_number)].copy()

train_texts = train_df["message"].fillna("").astype(str).to_numpy()
valid_texts = valid_df["message"].fillna("").astype(str).to_numpy()
train_labels = train_df["label"].map(label_to_number).astype("int32").to_numpy()
valid_labels = valid_df["label"].map(label_to_number).astype("int32").to_numpy()

print(f"Training messages: {len(train_texts):,}")
print(f"Validation messages: {len(valid_texts):,}")
print("Training class distribution:")
print(train_df["label"].value_counts())

# Convert raw SMS text into TF-IDF word and bigram features.
max_tokens = 12000
text_vectorizer = keras.layers.TextVectorization(
    max_tokens=max_tokens,
    standardize="lower_and_strip_punctuation",
    split="whitespace",
    ngrams=2,
    output_mode="tf_idf"
)
text_vectorizer.adapt(train_texts)

# Build an actual neural network classifier.
model = keras.Sequential([
    keras.Input(shape=(), dtype=tf.string),
    text_vectorizer,
    keras.layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=keras.regularizers.l2(1e-5)
    ),
    keras.layers.Dropout(0.4),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.25),
    keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc")
    ]
)

model.summary()

# Use tf.data pipelines for efficient batching and training.
batch_size = 32
train_dataset = (
    tf.data.Dataset.from_tensor_slices((train_texts, train_labels))
    .shuffle(buffer_size=len(train_texts), seed=42)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)
valid_dataset = (
    tf.data.Dataset.from_tensor_slices((valid_texts, valid_labels))
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-5
    )
]

history = model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=25,
    callbacks=callbacks,
    verbose=1
)

# Evaluate on the supplied validation set.
validation_metrics = model.evaluate(valid_dataset, return_dict=True, verbose=0)
print("\nValidation metrics:")
for metric_name, metric_value in validation_metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

# Plot the training history.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="Training loss")
axes[0].plot(history.history["val_loss"], label="Validation loss")
axes[0].set_title("Model loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="Training accuracy")
axes[1].plot(history.history["val_accuracy"], label="Validation accuracy")
axes[1].set_title("Model accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

# Return [spam probability, predicted label], as required by the challenge.
def predict_message(pred_text):
    if not isinstance(pred_text, str):
        raise TypeError("pred_text must be a string")

    spam_probability = float(
        model.predict(tf.constant([pred_text]), verbose=0)[0][0]
    )
    spam_probability = float(np.clip(spam_probability, 0.0, 1.0))
    predicted_label = "spam" if spam_probability >= 0.5 else "ham"

    return [spam_probability, predicted_label]

pred_text = "how are you doing today?"
prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):



  return (prediction)

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
